# Azure AI Search Pipeline Setup — SharePoint

Build an Azure AI Search indexing pipeline for SharePoint Online using the Python SDK (`azure-search-documents`).

This performs the same operations as the REST API (.rest files 01–04), but via the Python SDK.

| Step | Description |
|------|-------------|
| 1 | **Data Source** — SharePoint Online (ACL ingestion: userIds + groupIds) |
| 2 | **Index** — Vector Search + Semantic Search + Permission Filter |
| 3 | **Skillset** — ContentUnderstandingSkill + Embeddings + Index Projections |
| 4 | **Indexer** — Run the pipeline |

### References
- [ContentUnderstandingSkill (Python SDK)](https://learn.microsoft.com/en-us/python/api/azure-search-documents/azure.search.documents.indexes.models.contentunderstandingskill?view=azure-python-preview)
- [Permission filtering in Azure AI Search](https://learn.microsoft.com/azure/search/search-security-trimming-for-azure-search)

In [ ]:
# The preview version of azure-search-documents is required
# (ContentUnderstandingSkill is a preview API)
%pip install azure-search-documents --pre azure-identity python-dotenv --quiet

## Configuration

Set up connection details and resource names.  
API keys are loaded from environment variables or a `.env` file.

> **Note**: Add `SHAREPOINT_CONNECTION_STRING` to `.env`:  
> `SharePointOnlineEndpoint=https://...;ApplicationId=...;ApplicationSecret=...;TenantId=...;`

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, get_bearer_token_provider
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    # Index
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    # Vector search
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    ScalarQuantizationCompression,
    ScalarQuantizationParameters,
    RescoringOptions,
    # Semantic search
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
    # Data source
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection,
    # Skillset skills
    ContentUnderstandingSkill,
    ContentUnderstandingSkillChunkingProperties,
    AzureOpenAIEmbeddingSkill,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    # Skillset structure
    SearchIndexerSkillset,
    AIServicesAccountKey,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    IndexProjectionMode,
    # Indexer
    SearchIndexer,
    FieldMapping,
)
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

load_dotenv(override=True)

# ── Azure AI Search ──────────────────────────────────────
search_endpoint = os.environ.get("AZURE_SEARCH_ENDPOINT", "https://demo-search01.search.windows.net")
search_api_key  = os.environ["AZURE_SEARCH_API_KEY"]

# ── Credentials ───────────────────────────────────────────
tenant_id = os.environ.get("AZURE_TENANT_ID")
credential = AzureCliCredential(tenant_id=tenant_id)  # Entra: for user token (permission filter)
if search_api_key:
    search_credential = AzureKeyCredential(search_api_key)  # API key: for AI Search management
else:
    search_credential = credential

# ── Azure OpenAI ─────────────────────────────────────────
aoai_endpoint        = os.environ.get("AZURE_OPENAI_ENDPOINT", "https://demo-aifoundry02.openai.azure.com")
aoai_api_key         = os.environ["AZURE_OPENAI_API_KEY"]
embedding_deployment = "text-embedding-3-large"
embedding_model      = "text-embedding-3-large"
embedding_dimensions = 3072

# ── AI Services (for ContentUnderstandingSkill) ──────────
ai_services_key = os.environ["AI_SERVICES_KEY"]
ai_services_url = os.environ["AI_SERVICES_SUBDOMAIN_URL"]

# ── SharePoint ────────────────────────────────────────────
sharepoint_connection_string = os.environ["SHAREPOINT_CONNECTION_STRING"]

# ── Resource Names ────────────────────────────────────────
name_prefix     = os.environ.get("NAME_PREFIX", "ks-sharepoint-test")
datasource_name = f"{name_prefix}-datasource"
index_name      = f"{name_prefix}-index"
skillset_name   = f"{name_prefix}-skillset"
indexer_name    = f"{name_prefix}-indexer"

print("Configuration loaded.")
print(f"  Search endpoint : {search_endpoint}")
print(f"  AOAI endpoint   : {aoai_endpoint}")
print(f"  Name prefix     : {name_prefix}")
print(f"  Index name      : {index_name}")

## Step 1 — Create Data Source

Define the SharePoint Online connection with ACL ingestion enabled.  
The connection string includes the SharePoint endpoint, Application ID, Application Secret, and Tenant ID.

`indexerPermissionOptions: ["userIds", "groupIds"]` enables the indexer to capture ACL metadata  
(Entra user OIDs and group OIDs) from SharePoint and store them in the index for query-time permission filtering.

In [ ]:
indexer_client = SearchIndexerClient(endpoint=search_endpoint, credential=search_credential)

data_source = SearchIndexerDataSourceConnection(
    name=datasource_name,
    description=f"Data source for knowledge source '{name_prefix}'",
    type="sharepoint",
    connection_string=sharepoint_connection_string,
    container=SearchIndexerDataContainer(name="allSiteLibraries"),
    indexer_permission_options=["userIds", "groupIds"]
)

result = indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source '{result.name}' created or updated.")

## Step 2 — Create Index

Create the search index with permission filtering enabled.

| Field | Purpose |
|-------|---------|
| `uid` | Document key (keyword analyzer) |
| `snippet_parent_id` | Parent document ID for text chunks |
| `metadata_spo_item_path` | Source file URL (SharePoint item path) |
| `snippet` | Text content (ja.microsoft analyzer) |
| `snippet_vector` | Vector embedding (3072 dimensions) |
| `UserIds` | ACL: Entra user OIDs with access (`permissionFilter: userIds`) |
| `GroupIds` | ACL: Entra group OIDs with access (`permissionFilter: groupIds`) |

Vector Search: HNSW + Scalar Quantization (int8) + Azure OpenAI Vectorizer  
Semantic Search: snippet field set as prioritized content  
Permission Filter: `permissionFilterOption: enabled` — queries filtered by the caller's Entra identity

In [ ]:
index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)

# ── Field Definitions ─────────────────────────────────────
fields = [
    SearchField(
        name="uid", type=SearchFieldDataType.String,
        key=True, searchable=True, filterable=False,
        stored=True, sortable=True, facetable=False,
        analyzer_name="keyword",
    ),
    SearchField(
        name="snippet_parent_id", type=SearchFieldDataType.String,
        searchable=False, filterable=True,
        stored=True, sortable=False, facetable=False,
    ),
    SearchField(
        name="metadata_spo_item_path", type=SearchFieldDataType.String,
        searchable=False, filterable=True,
        stored=True, sortable=False, facetable=False,
    ),
    SearchField(
        name="snippet", type=SearchFieldDataType.String,
        searchable=True, filterable=False,
        stored=True, sortable=False, facetable=False,
        analyzer_name="ja.microsoft",
    ),
    SearchField(
        name="snippet_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True, filterable=False,
        stored=True, sortable=False, facetable=False,
        vector_search_dimensions=embedding_dimensions,
        vector_search_profile_name=f"{name_prefix}-vector-search-profile",
    ),
    SearchField(
        name="UserIds",
        type=SearchFieldDataType.Collection(SearchFieldDataType.String),
        searchable=False, filterable=True, hidden=True,
        stored=True, sortable=False, facetable=False,
        permission_filter="userIds",
    ),
    SearchField(
        name="GroupIds",
        type=SearchFieldDataType.Collection(SearchFieldDataType.String),
        searchable=False, filterable=True, hidden=True,
        stored=True, sortable=False, facetable=False,
        permission_filter="groupIds",
    ),
]

# ── Vector Search Configuration ───────────────────────────
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name=f"{name_prefix}-vector-search-algorithm",
            parameters=HnswParameters(metric="cosine", m=4, ef_construction=400, ef_search=500),
        ),
    ],
    profiles=[
        VectorSearchProfile(
            name=f"{name_prefix}-vector-search-profile",
            algorithm_configuration_name=f"{name_prefix}-vector-search-algorithm",
            vectorizer_name=f"{name_prefix}-vectorizer",
            compression_name=f"{name_prefix}-vector-search-scalar-quantization",
        ),
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            vectorizer_name=f"{name_prefix}-vectorizer",
            parameters=AzureOpenAIVectorizerParameters(
                resource_url=aoai_endpoint,
                deployment_name=embedding_deployment,
                model_name=embedding_model,
                api_key=aoai_api_key,
            ),
        ),
    ],
    compressions=[
        ScalarQuantizationCompression(
            compression_name=f"{name_prefix}-vector-search-scalar-quantization",
            parameters=ScalarQuantizationParameters(quantized_data_type="int8"),
            rescoring_options=RescoringOptions(
                enable_rescoring=True,
                default_oversampling=4.0,
                rescore_storage_method="preserveOriginals",
            ),
        ),
    ],
)

# ── Semantic Search Configuration ─────────────────────────
semantic_config = SemanticConfiguration(
    name=f"{name_prefix}-semantic-configuration",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="snippet")],
    ),
)
semantic_search = SemanticSearch(
    default_configuration_name=f"{name_prefix}-semantic-configuration",
    configurations=[semantic_config],
)

# ── Create Index ──────────────────────────────────────────
index = SearchIndex(
    name=index_name,
    description=f"Search index for knowledge source '{name_prefix}'",
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
    permission_filter_option="enabled",
)

result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' created or updated.")

## Step 3 — Create Skillset

Create the skillset, composed of the following 2 skills:

| # | Skill | Description |
|---|-------|-------------|
| 1 | **ContentUnderstandingSkill** | Extract text chunks and images from documents |
| 2 | **AzureOpenAIEmbeddingSkill** | Generate vector embeddings for text chunks |

Index Projections project text chunks (with ACL fields `UserIds`/`GroupIds`) into the index.  
`extraction_options: ["images", "locationMetadata"]` enables image extraction and location metadata.

In [ ]:
# ── Skill 1: ContentUnderstandingSkill ────────────────────────────────
content_understanding_skill = ContentUnderstandingSkill(
    name="contentUnderstandingSkill",
    context="/document",
    extraction_options=["images", "locationMetadata"],
    inputs=[
        InputFieldMappingEntry(name="file_data", source="/document/file_data"),
    ],
    outputs=[
        OutputFieldMappingEntry(name="text_sections", target_name="text_sections"),
        OutputFieldMappingEntry(name="normalized_images", target_name="normalized_images"),
    ],
    chunking_properties=ContentUnderstandingSkillChunkingProperties(
        unit="characters",
        maximum_length=2000,
        overlap_length=200,
    ),
)

# ── Skill 2: Embedding for Text Chunks ───────────────────────────────
text_embedding_skill = AzureOpenAIEmbeddingSkill(
    name="AzureOpenAIEmbeddingSkill",
    description="Generate embeddings",
    context="/document/text_sections/*",
    resource_url=aoai_endpoint,
    api_key=aoai_api_key,
    deployment_name=embedding_deployment,
    model_name=embedding_model,
    dimensions=embedding_dimensions,
    inputs=[
        InputFieldMappingEntry(name="text", source="/document/text_sections/*/content"),
    ],
    outputs=[
        OutputFieldMappingEntry(name="embedding", target_name="text_vector"),
    ],
)

# ── Index Projections ─────────────────────────────────────────────────
index_projections = SearchIndexerIndexProjection(
    selectors=[
        SearchIndexerIndexProjectionSelector(
            target_index_name=index_name,
            parent_key_field_name="snippet_parent_id",
            source_context="/document/text_sections/*",
            mappings=[
                InputFieldMappingEntry(
                    name="snippet_vector",
                    source="/document/text_sections/*/text_vector",
                ),
                InputFieldMappingEntry(
                    name="snippet",
                    source="/document/text_sections/*/content",
                ),
                InputFieldMappingEntry(
                    name="metadata_spo_item_path",
                    source="/document/metadata_spo_item_path",
                ),
                InputFieldMappingEntry(
                    name="UserIds",
                    source="/document/metadata_user_ids",
                ),
                InputFieldMappingEntry(
                    name="GroupIds",
                    source="/document/metadata_group_ids",
                ),
            ],
        ),
    ],
    parameters=SearchIndexerIndexProjectionsParameters(
        projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS,
    ),
)

# ── AI Services (AIServicesAccountKey + subdomainUrl) ─────────────────
cognitive_services_account = AIServicesAccountKey(
    key=ai_services_key,
    subdomain_url=ai_services_url,
    description="AI Services for knowledge source",
)

# ── Create Skillset ───────────────────────────────────────────────────
skillset = SearchIndexerSkillset(
    name=skillset_name,
    description="Skillset for knowledge source 'ks-sharepoint-test'",
    skills=[
        content_understanding_skill,
        text_embedding_skill,
    ],
    cognitive_services_account=cognitive_services_account,
    index_projection=index_projections,
)

result = indexer_client.create_or_update_skillset(skillset)
print(f"Skillset '{result.name}' created or updated.")

## Step 4 — Create and Run the Indexer

Create and run the indexer.  
`allowSkillsetToReadFileData: true` enables the skillset to read file data (e.g., images).

In [ ]:
indexer = SearchIndexer(
    name=indexer_name,
    description="Indexer for knowledge source 'ks-sharepoint-test'",
    data_source_name=datasource_name,
    skillset_name=skillset_name,
    target_index_name=index_name,
    parameters={
        "maxFailedItems": -1,
        "configuration": {
            "allowSkillsetToReadFileData": True,
        },
    },
    field_mappings=[
        FieldMapping(
            source_field_name="metadata_user_ids",
            target_field_name="UserIds",
        ),
        FieldMapping(
            source_field_name="metadata_group_ids",
            target_field_name="GroupIds",
        ),
    ],
)

result = indexer_client.create_or_update_indexer(indexer)
print(f"Indexer '{result.name}' created or updated.")
print("Indexer is running. Please wait a few minutes before querying.")

In [ ]:
# Check indexer execution status
status = indexer_client.get_indexer_status(indexer_name)
print(f"Status          : {status.status}")
print(f"Last result     : {status.last_result.status if status.last_result else 'N/A'}")
if status.last_result:
    print(f"  Items succeeded : {status.last_result.item_count}")
    print(f"  Items failed    : {status.last_result.failed_item_count}")
    print(f"  Start time      : {status.last_result.start_time}")
    print(f"  End time        : {status.last_result.end_time}")
    if status.last_result.errors:
        for err in status.last_result.errors:
            print(f"  Error: {err.error_message}")

## Search Test

After the indexer finishes, test with vector search + semantic search.  
Permission filtering is applied via `x-ms-query-source-authorization: Bearer <token>`,  
which scopes results to documents the signed-in user has access to in SharePoint.

In [ ]:
query = "What is the cruise control in Prius?"

user_token = get_bearer_token_provider(credential, "https://search.azure.com/.default")()

search_client = SearchClient(
    endpoint=search_endpoint,
    credential=search_credential,
    index_name=index_name,
)
vector_query = VectorizableTextQuery(
    text=query,
    k=50,
    fields="snippet_vector",
)

results = search_client.search(
    search_text=query,
    vector_queries=[vector_query],
    select=["snippet", "metadata_spo_item_path"],
    top=3,
    headers={"x-ms-query-source-authorization": user_token},
)

for i, result in enumerate(results, 1):
    score = result["@search.score"]
    path = result.get("metadata_spo_item_path", "N/A")
    snippet = result.get("snippet", "")
    print(f"\n--- Result {i} (score: {score:.4f}) ---")
    print(f"metadata_spo_item_path: {path}")
    print(f"snippet : {snippet[:300]}..." if len(snippet) > 300 else f"snippet : {snippet}")

### Run 101 and onward